# 📦 Stash the CPC corpus to Google Drive (one paid scan, then free forever)

**What this does:** ONE ~$10 BigQuery scan pulls the full CPC corpus (G06F16/N/40, ~728K patents) + 26 force-included patents into a permanent BigQuery table, then exports it to **Google Drive as Parquet**. After this you build the index from Drive for **$0** (`--from-cache`).

**Cost safety:** the scan is hard-capped at 2 TB (errors at \$0 if exceeded). Cell 7 is a free dry-run gate — read the cost, then run Cell 8.

The \$10 is *banked* by Cell 8 (permanent table). If the Drive export (Cell 9) hiccups, just re-run it — it reads the banked table for ~\$0.10, never re-scans.


## 1 · Install


In [ ]:
!pip install -q google-cloud-bigquery google-cloud-bigquery-storage pyarrow


## 2 · Auth + project


In [ ]:
from google.colab import auth
auth.authenticate_user()
PROJECT = 'ss-fleet-498508'
print('Authed:', PROJECT)


## 3 · Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/patent_kb_stash/cpc_corpus'
import os; os.makedirs(OUT, exist_ok=True)
print('Stash dir:', OUT)


## 4 · Get the pipeline code


In [ ]:
import subprocess
BRANCH='feature/m0-multi-rule-ingestion'
!rm -rf google-patent-kb
subprocess.run(['git','clone','--branch',BRANCH,'--depth','1','https://github.com/blazingbunny/google-patent-kb.git'], check=True)
%cd google-patent-kb


## 5 · Build the stash query
Full CPC corpus ∪ 26 force-includes. **No --limit** (we want all of it). No inventor rule (they're inside the CPC fence already).


In [ ]:
import pipeline
PUB_NUMBERS = ["US9449105B1", "US8577893B1", "US20110040768A1", "US7974964B2", "US9245022B2", "US9767157B2", "US8209331B1", "US8051076B1", "US10650005B2", "US8762370B1", "US20190026338A1", "US9213748B1", "US12141132B1", "US9959315B1", "US9031929B1", "US10180964B1", "US10055467B1", "US12013887B2", "US9436747B1", "US8612453", "US11836177B2", "US10922326B2", "US9477711B2", "US11720577B2", "US10068022B2", "US9009192B1"]
cfg = pipeline.Config(project=PROJECT, cpc=['G06F16','G06N','G06F40'],
                      publication_numbers=PUB_NUMBERS)  # no limit, no inventors
print('Query built. Cost cap:', cfg.max_bytes_billed/1e9, 'GB')


## 6 · Prepare BigQuery params


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)
Q = pipeline.build_query(cfg)
P = pipeline.build_query_params(cfg)
BQP = []
for k,v in P.items():
    BQP.append(bigquery.ArrayQueryParameter(k,'STRING',v) if isinstance(v,list) else bigquery.ScalarQueryParameter(k,'STRING',v))
print('params ready')


## 7 · 🚦 DRY-RUN COST GATE (free)
Validates SQL + prints cost. **Billed \$0.** Read it, then run Cell 8.


In [ ]:
jc = bigquery.QueryJobConfig(query_parameters=BQP, dry_run=True, maximum_bytes_billed=cfg.max_bytes_billed)
j = client.query(Q, job_config=jc)
gb = j.total_bytes_processed/1e9
print('✅ DRY RUN OK — valid SQL.')
print(f'   Scan: {gb:,.1f} GB   Est. cost: ${max(0,(gb-1000)/1000*6.25):.2f} (after 1TB/mo free tier; ${gb/1000*6.25:.2f} if no free tier left)')
print('\n>>> If OK, run Cell 8 (the ONE paid scan). <<<')


## 8 · ▶️ SCAN → permanent BigQuery table (the ~$10, banked)
Writes the corpus to `patent_kb_stash.cpc_corpus`. This is the only paid step.


In [ ]:
import time
ds = bigquery.Dataset(f'{PROJECT}.patent_kb_stash'); ds.location='US'
client.create_dataset(ds, exists_ok=True)
TABLE_ID = f'{PROJECT}.patent_kb_stash.cpc_corpus'
jc = bigquery.QueryJobConfig(query_parameters=BQP, destination=TABLE_ID,
        write_disposition='WRITE_TRUNCATE', maximum_bytes_billed=cfg.max_bytes_billed)
job = client.query(Q, job_config=jc)
print('Scanning... (this is the paid step)')
while not job.done():
    job.reload(); time.sleep(3)
if job.errors: raise RuntimeError(job.errors)
t = client.get_table(TABLE_ID)
print(f'✅ Done. Billed: {(job.total_bytes_billed or 0)/1e9:,.1f} GB  (~${(job.total_bytes_billed or 0)/1e12*6.25:.2f})')
print(f'   Banked {t.num_rows:,} patents in {TABLE_ID}')


## 9 · 💾 Export table → Drive (Parquet shards) — memory-safe + resumable
Processes 10K patents at a time (Colab RAM-safe) and **skips already-written shards** — if it crashes, just re-run this cell; it continues, never re-scans. If it still OOMs, drop SHARD to 5000.


In [ ]:
import os, glob, gc
import pyarrow.parquet as pq
SHARD = 10000
t = client.get_table(TABLE_ID)
total = t.num_rows
existing = sorted(glob.glob(os.path.join(OUT, 'cpc_*.parquet')))
done  = sum(pq.ParquetFile(f).metadata.num_rows for f in existing)
shard = len(existing)
print(f'Resuming from {done:,}/{total:,}  ({shard} shards already on Drive)')
while done < total:
    batch = client.list_rows(t, start_index=done, max_results=SHARD, page_size=5000)
    tbl = batch.to_arrow()
    if tbl.num_rows == 0:
        break
    pq.write_table(tbl, os.path.join(OUT, f'cpc_{shard:04d}.parquet'), compression='snappy')
    done  += tbl.num_rows
    shard += 1
    print(f'  {done:,}/{total:,} written ({shard} shards)')
    del tbl, batch; gc.collect()
print(f'✅ Export complete: {done:,} patents in {shard} shards at {OUT}')


## 10 · ✅ Verify the Drive stash


In [ ]:
import glob, pyarrow.parquet as pq
files = sorted(glob.glob(OUT+'/*.parquet'))
n = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
sz = sum(os.path.getsize(f) for f in files)/1e9
print(f'{len(files)} shards | {n:,} patents | {sz:.1f} GB on Drive')
print('STASH PATH (use with --from-cache):', OUT)


## 11 · 🧹 (Optional) Delete the BQ table
Run **only after Cell 10 confirms the Drive copy**. Stops ~\$1.60/mo BQ storage. Safe — your data is on Drive.


In [ ]:
client.delete_table(TABLE_ID, not_found_ok=True)
print('Deleted', TABLE_ID, '- stash lives on Drive now.')


## ♻️ How to build the index later — for $0
In any future Colab (GPU), clone the branch and run:
```python
import subprocess
subprocess.run(['python','pipeline.py',
   '--from-cache','/content/drive/MyDrive/patent_kb_stash/cpc_corpus',
   '--device','cuda','--batch-size','32',
   '--output','./data/full'], check=True)
```
No BigQuery, no scan, **$0** — rebuild as many times as you want.
